In [ ]:
# imports
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime

In [ ]:
# MUST ADD CALIBRATED DATA TO THIS!!!
# use data_processing.ipynb BEFORE USING THIS

In [ ]:
# variables
filename = 'nothing right now'
df = pd.read_csv(filename, header=None, names = ["time", "T0", "T1", "T2", "T3"])
df.head()
c_drag = 0.1
r = 0.25
mass = 50/1000

In [ ]:
# constants
g = 9.8
rho_air = 1.225
rho_helium = 0.179
mu = 0.0000181
# mu is dynamic viscosity of air
# theta is angle from vertical downward direction
# beta is angle in horizontal plane
theta = math.radians(55)
beta1 = 0
beta2 = math.pi/2
beta3 = math.pi
beta4 = 3 * math.pi/2

betas = [beta1, beta2, beta3, beta4]

In [ ]:
# first, calculating buoyancy and lift

volume = (4/3) * math.pi * r**3
m_helium = rho_helium * volume
# Solving buoyancy and weight
f_buoyancy = (rho_air) * g * volume
# add mass of helium
f_weight = (mass + m_helium) * g


# add lift
f_lift = np.array([0, 0, f_buoyancy - f_weight])

In [ ]:
vectorized_df = df[["time"]].copy()

# calculating tension and drag

def tension_vector(T, theta, beta):
  ''' vectorizes measured tension '''
  T_vec = np.array([
        [T * math.sin(theta) * math.cos(beta)],
        [T * math.sin(theta) * math.sin(beta)],
        [-T * math.cos(theta)]
  ])

  return T_vec

def vectorize_tensions(df, theta, beta)
  ''' applys tension_vector to every row of dataframe
  saves each tether vector in new dataframe'''
  
  tension_cols = ["T0", "T1", "T2", "T3"]
  
  for i, col in enumerate(tension_cols):
    beta = betas[i]
        
    vectorized_df[f"Tension{i}"] = df[col].apply(
    lambda T: tension_vector(T, theta, beta)
    )   
  
  return vectorized_df

vectorized_df = vectorize_tensions(df, theta, betas)

vectorized_df.head()

In [ ]:
# calculating drag vector

def drag_vector(vectorized_df, f_lift):
    """
    Creates new columns for total tension, drag vector, drag direction,
    and drag magnitude.

    Force equilibrium:

        F_drag + total_tension + f_lift = 0

    So:

        F_drag = -(total_tension + f_lift)
    """

    vectorized_df["total_tension"] = (
        vectorized_df["Tension0"] +
        vectorized_df["Tension1"] +
        vectorized_df["Tension2"] +
        vectorized_df["Tension3"]
    )

    vectorized_df["drag"] = vectorized_df["total_tension"].apply(
        lambda T_total: -(T_total + f_lift)
    )

    vectorized_df["drag_magnitude"] = vectorized_df["drag"].apply(
        lambda drag: np.linalg.norm(drag)
    )

    vectorized_df["drag_direction"] = vectorized_df.apply(
        lambda row: row["drag"] / row["drag_magnitude"]
        if row["drag_magnitude"] != 0
        else np.array([0, 0, 0]),
        axis=1
    )

    return vectorized_df

vectorized_df = drag_vector(vectorized_df, f_lift)
vectorized_df.head()

In [ ]:
# calculating velocity

def velocity_mag(vectorized_df, c_drag, rho_air, r):
    ''' finds velocity magnitude from drag magnitude'''
    area = math.pi * r**2

    vectorized_df["velocity_magnitude"] = np.sqrt(
        (2 * vectorized_df["drag_magnitude"]) / (c_drag * rho_air * area)
    )

    return vectorized_df

def velocity_vector(vectorized_df):
    ''' finds velocity vector '''
    if "drag_direction" not in vectorized_df.columns:
        raise ValueError("drag_direction column is missing. Please run drag_vector first.")

    if "velocity_magnitude" not in vectorized_df.columns:
        raise ValueError("velocity_magnitude column is missing. Please run velocity_mag first.")

    vectorized_df["velocity_vector"] = vectorized_df.apply(
        lambda row: row["velocity_magnitude"] * row["drag_direction"],
        axis=1
    )

    return vectorized_df

vectorized_df = velocity_mag(vectorized_df, c_drag, rho_air, r)
vectorized_df = velocity_vector(vectorized_df)

vectorized_df.head()

In [ ]:
# finding drag angles

def find_angles(vectorized_df):
    """
    Finds the angles of the drag vector.

    alpha is angle in the horizontal xy-plane
    phi is vertical angle relative to the xy-plane
    """

    vectorized_df["alpha"] = vectorized_df["drag"].apply(
        lambda drag: math.atan2(drag[1], drag[0])
    )

    vectorized_df["phi"] = vectorized_df["drag"].apply(
        lambda drag: math.atan2(
            drag[2],
            math.sqrt(drag[0]**2 + drag[1]**2)
        )
    )

    return vectorized_df

vectorized_df = find_angles(vectorized_df)

vectorized_df.head()

In [ ]:
# -----------------------------
# graph velocity magnitude over time
# -----------------------------

plt.plot(vectorized_df["time"], vectorized_df["velocity_magnitude"])
plt.xlabel("Time")
plt.ylabel("Velocity Magnitude (m/s)")
plt.title("Velocity Magnitude Over Time")
plt.grid(True)
plt.show()

In [ ]:
# -----------------------------
# save processed data
# -----------------------------

output_file_name = f"v_vs_t_{filename}"
vectorized_df.to_csv(output_file_name, index=False)
